# Zalo AI Challenge - Datamate & ZAIC Hybrid Inference trên Google Colab

Notebook này được thiết kế để **chạy suy luận trực tiếp trên tập TEST** trên môi trường Google Colab:
- **Clone trực tiếp mã nguồn từ GitHub**: Không cần tải repo lên Google Drive.
- **Mount Google Drive**: Dùng để lấy tập test (nếu có) và **tự động lưu toàn bộ file kết quả (`submission.jsonl`), báo cáo và logs chi tiết** về Google Drive.
- **Tích hợp Xử lý đầu vào & `as_of_day` từ ZAIC_2026**: Hỗ trợ đa định dạng dữ liệu (CSV, Excel, Parquet, SQLite), DuckDB tối ưu và bộ phân tích thời gian tiếng Việt chuyên sâu (`viet_time.py`, `describe_period`).

## 1. Cấu hình & Clone Repo từ GitHub, Mount Google Drive

In [1]:
import os, sys, subprocess, shutil, getpass
from pathlib import Path
from urllib.parse import urlparse

IN_COLAB = 'google.colab' in sys.modules

# ======================================================================
# 1. THIẾT LẬP REPO GITHUB CỦA BẠN (Thay đổi nếu dùng repo riêng)
# ======================================================================
REPO_URL = "https://github.com/TRUNGNT0/zalo_ai_2026_track2"  # <-- URL repo github
BRANCH = "main"
TARGET_DIR = Path("/content/datamate_repo")

if IN_COLAB:
    # Mount Google Drive
    print("▶ Đang kết nối Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Clone / Pull repo từ GitHub
    print("▶ Đang lấy mã nguồn từ GitHub...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")  # Token từ Colab Secrets (nếu repo private)
    except Exception:
        pass

    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    if (TARGET_DIR / ".git").is_dir():
        print(f"Thư mục {TARGET_DIR} đã tồn tại. Đang cập nhật code mới nhất...")
        subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET_DIR, check=True)
    else:
        print(f"Đang clone {REPO_URL} (branch: {BRANCH})...")
        res = subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            if "Authentication failed" in res.stderr or "could not read Username" in res.stderr:
                token = getpass.getpass("Repo private! Hãy nhập GitHub Personal Access Token (PAT): ")
                clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
                subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], check=True)
            else:
                raise RuntimeError(f"Lỗi clone Git: {res.stderr}")
    ROOT = TARGET_DIR
else:
    ROOT = Path.cwd().resolve()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"✓ Thư mục làm việc: {ROOT}")

▶ Đang kết nối Google Drive...
Mounted at /content/drive
▶ Đang lấy mã nguồn từ GitHub...
Đang clone https://github.com/TRUNGNT0/zalo_ai_2026_track2 (branch: main)...
✓ Thư mục làm việc: /content/datamate_repo


## 2. Cài đặt các thư viện phụ thuộc & Kiểm tra GPU

In [2]:
if IN_COLAB:
    print("▶ Đang cài đặt thư viện cần thiết...")
    !pip install -q duckdb pandas numpy openpyxl xlrd pyarrow python-calamine jsonschema python-dotenv matplotlib
    !pip install -q transformers accelerate peft bitsandbytes

import torch
print(f"PyTorch: {torch.__version__} | CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {device_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️ Lưu ý: Đang chạy trên CPU, hãy đổi Runtime sang GPU (T4, L4 hoặc A100) để suy luận nhanh.")

▶ Đang cài đặt thư viện cần thiết...
PyTorch: 2.14.1+cu130 | CUDA khả dụng: True
GPU: NVIDIA A100-SXM4-80GB (79.3 GB VRAM)


## 3. Thiết lập Thư mục Lưu Kết Quả & Logs trên Google Drive

In [3]:
from datetime import datetime

# ĐỊNH NGHĨA ĐƯỜNG DẪN LƯU KẾT QUẢ TRÊN GOOGLE DRIVE
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if IN_COLAB:
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ZaloAI_Outputs") / f"run_{timestamp}"
else:
    DRIVE_OUTPUT_DIR = ROOT / "outputs" / f"run_{timestamp}"

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)

# Thư mục tạm tốc độ cao trên Colab SSD
LOCAL_RUN_DIR = Path("/content/run_test_tmp" if IN_COLAB else ROOT / "outputs" / "tmp")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)

print(f"✓ Thư mục đích lưu kết quả & logs trên Drive: {DRIVE_OUTPUT_DIR}")
print(f"✓ Thư mục chạy tạm thời: {LOCAL_RUN_DIR}")

def sync_to_drive():
    """Đồng bộ ngay lập tức submission.jsonl và toàn bộ logs về Google Drive."""
    if not IN_COLAB:
        return
    for item in LOCAL_RUN_DIR.rglob("*"):
        if item.is_file() and not item.name.endswith(".tmp") and not item.name.endswith(".wal"):
            rel = item.relative_to(LOCAL_RUN_DIR)
            dest = DRIVE_OUTPUT_DIR / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest)
    print(f"[Sync] Đã sao lưu kết quả & logs mới nhất về Google Drive: {DRIVE_OUTPUT_DIR}", flush=True)

✓ Thư mục đích lưu kết quả & logs trên Drive: /content/drive/MyDrive/ZaloAI_Outputs/run_20261115_075525
✓ Thư mục chạy tạm thời: /content/run_test_tmp


## 4. Xác định Tập Dữ Liệu Test (Public Test hoặc Private Test)

Tìm kiếm thư mục chứa tập test trên repo hoặc trên Google Drive.

In [4]:
# Tìm kiếm tập test tại các đường dẫn khả dĩ
CANDIDATE_PATHS = [
    ROOT / "dataset" / "public_test",
    ROOT / "data" / "public_test",
    Path("/content/drive/MyDrive/ZaloAI/public_test"),
    Path("/content/drive/MyDrive/dataset/public_test"),
    ROOT / "dataset" / "test",
]

TEST_DIR = None
for p in CANDIDATE_PATHS:
    if p.is_dir() and (p / "questions.jsonl").is_file():
        TEST_DIR = p
        break

# Hoặc bạn có thể chỉ định trực tiếp đường dẫn tập test tại đây:
# TEST_DIR = Path("/content/drive/MyDrive/duong_dan_tap_test")

if TEST_DIR is None:
    print("⚠️ Không tự động tìm thấy questions.jsonl trong các đường dẫn mặc định.")
    print("Hãy đặt đường dẫn tập test vào biến TEST_DIR bên dưới.")
    TEST_DIR = ROOT / "dataset" / "public_test"
else:
    print(f"✓ Đã tìm thấy tập test tại: {TEST_DIR}")

# Kiểm tra số lượng câu hỏi và cấu trúc as_of_date / as_of_day
import json
q_file = TEST_DIR / "questions.jsonl"
if q_file.is_file():
    questions = [json.loads(line) for line in q_file.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    print(f"✓ Tổng số câu hỏi test: {len(questions)}")
    sample = questions[0]
    print(f"✓ Mẫu câu hỏi [0]: ID={sample.get('id')}, Biz={sample.get('business_id')}")
    as_of = sample.get('as_of_date') or sample.get('as_of_day')
    print(f"✓ as_of_date / as_of_day: {as_of}")
    print(f"  Nội dung: {sample.get('question')}")

✓ Đã tìm thấy tập test tại: /content/datamate_repo/dataset/public_test
✓ Tổng số câu hỏi test: 1000
✓ Mẫu câu hỏi [0]: ID=q_008fdebb6ec9, Biz=dev_002
✓ as_of_date / as_of_day: 2026-05-24
  Nội dung: Tháng 11 năm 2025, nhóm trà có mặt hàng nào bán được nhiều ly nhất?


## 5. Khởi Tạo Pipeline với Xử Lý Đầu Vào & `as_of_day` từ ZAIC_2026

Pipeline sử dụng:
- Xử lý đầu vào: Chuẩn hóa dữ liệu thô, nạp vào DuckDB với kiểm tra schema và UDFs (`vn_date`, `vn_date_year`, `vn_norm`).
- `as_of_day`: Bộ phân tích thời gian chuyên sâu `viet_time.py` & `describe_period` để xác định chính xác khoảng thời gian của câu hỏi.

In [5]:
from datamate.local_pipeline import LocalPipelineConfig, LocalSQLPipeline
from datamate.tools.timeutil import describe_period
import inspect

# Kiểm tra nhanh bộ phân tích thời gian ZAIC
test_hint = describe_period("tháng này doanh thu bao nhiêu", "2026-05-25")
print(f"✓ Phân tích thời gian ZAIC cho 'tháng này': {test_hint}")

# Cấu hình model & pipeline
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # hoặc Qwen/Qwen3.5-9B
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0.0
hf_quant = "none" if vram_gb >= 40.0 else "auto"  # T4/V100/L4 dùng 4-bit, A100 dùng full/bf16

cfg_kwargs = dict(
    dataset_dir=str(TEST_DIR.parent),
    out_dir=str(LOCAL_RUN_DIR),
    model=MODEL_NAME,
    hf_quant=hf_quant,
    max_model_len=16384,
    sql_tokens=2048,
    manifest_tokens=3072,
    n_samples=3,
    agree_min=2,
    max_retry=2,
    generation_timeout=600.0,
)

sig = inspect.signature(LocalPipelineConfig)
if "ingestion_mode" in sig.parameters:
    cfg_kwargs["ingestion_mode"] = "direct"
if "require_prepared_database" in sig.parameters:
    cfg_kwargs["require_prepared_database"] = False

pipeline_config = LocalPipelineConfig(**cfg_kwargs)
pipeline = LocalSQLPipeline(pipeline_config)
if hasattr(pipeline, "base"):
    pipeline.base.ingestion_mode = "direct"
    pipeline.base.require_prepared_database = False

print("✓ Khởi tạo LocalSQLPipeline thành công!")

✓ Phân tích thời gian ZAIC cho 'tháng này': "tháng này" = 2026-05-01 đến 2026-05-25
✓ Khởi tạo LocalSQLPipeline thành công!


## 6. Nạp Dữ Liệu Shop vào DuckDB (Direct / Layered Ingestion)

In [6]:
from time import perf_counter

split_name = TEST_DIR.name
t0 = perf_counter()
print(f"▶ Đang chuẩn bị dữ liệu đầu vào cho split '{split_name}'...", flush=True)
prep_report = pipeline.prepare_datasets(splits=(split_name,))
print(f"✓ Chuẩn bị dữ liệu xong sau {perf_counter() - t0:.1f}s!")

# Kiểm tra database
validation = pipeline.validate_databases(splits=(split_name,))
print(f"✓ Đã xác minh thành công {len(validation)} databases shop trong DuckDB.")

# Đồng bộ báo cáo chuẩn bị dữ liệu sang Google Drive
sync_to_drive()

▶ Đang chuẩn bị dữ liệu đầu vào cho split 'public_test'...
Tải Qwen/Qwen2.5-Coder-7B-Instruct về ./models/Qwen__Qwen2.5-Coder-7B-Instruct (cần bật Internet trong Settings của notebook) ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[hf] /content/datamate_repo/models/Qwen__Qwen2.5-Coder-7B-Instruct | GPU 85GB | nạp full | torch.bfloat16 | lai linear-attention: False


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[direct-ingest] dev_001/README.md: Unsupported source format
[direct-ingest] dev_002/README.md: Unsupported source format
[direct-ingest] dev_003/README.md: Unsupported source format
[direct-ingest] dev_004/README.md: Unsupported source format
[direct-ingest] dev_005/README.md: Unsupported source format
[direct-ingest] dev_006/README.md: Unsupported source format
[direct-ingest] dev_007/README.md: Unsupported source format
[direct-ingest] dev_008/README.md: Unsupported source format
[direct-ingest] dev_009/README.md: Unsupported source format
[direct-ingest] dev_010/README.md: Unsupported source format
[direct-ingest] dev_011/README.md: Unsupported source format
[direct-ingest] dev_012/README.md: Unsupported source format
[direct-ingest] dev_013/README.md: Unsupported source format
[direct-ingest] dev_014/README.md: Unsupported source format
✓ Chuẩn bị dữ liệu xong sau 319.5s!
Validated 14 DuckDB databases
✓ Đã xác minh thành công 14 databases shop trong DuckDB.
[Sync] Đã sao lưu kết q

In [7]:
# Sync nền: copy log/cache nhẹ về Drive mỗi 5 phút để session đứt không mất kết quả,
# và khôi phục cache của lần chạy trước (nếu có) để chạy tiếp.
import threading, time

if IN_COLAB:
    (LOCAL_RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)
    for old in sorted(DRIVE_OUTPUT_DIR.parent.glob("run_*/logs/llm_cache_*.jsonl")):
        if old.parent.parent == DRIVE_OUTPUT_DIR:
            continue
        dest = LOCAL_RUN_DIR / "logs" / old.name
        if (not dest.exists()) or old.stat().st_mtime > dest.stat().st_mtime:
            shutil.copy2(old, dest)
            print(f"↺ Khôi phục cache: {old}")

    _SYNC_SUFFIX = {".jsonl", ".json", ".log", ".txt", ".csv"}
    def _bg_sync(period=300):
        while True:
            time.sleep(period)
            try:
                for f in LOCAL_RUN_DIR.rglob("*"):
                    if f.is_file() and f.suffix in _SYNC_SUFFIX:
                        dest = DRIVE_OUTPUT_DIR / f.relative_to(LOCAL_RUN_DIR)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(f, dest)
            except Exception as e:
                print(f"[bg-sync] lỗi: {e!r}", flush=True)
    threading.Thread(target=_bg_sync, daemon=True).start()
    print("✓ Đã bật sync nền (5 phút/lần)")

✓ Đã bật sync nền (5 phút/lần)


## 7. Chạy Suy Luận Toàn Bộ Tập Test & Tự Động Lưu Về Google Drive

In [8]:
# t_start = perf_counter()
# print(f"=== BẮT ĐẦU SUY LUẬN TRÊN TẬP {split_name} ===", flush=True)

# # Thực hiện dự đoán
# if hasattr(pipeline, f"predict_{split_name}"):
#     method = getattr(pipeline, f"predict_{split_name}")
#     submission_path = method()
# else:
#     # Fallback dự đoán tổng quát
#     submission_path = pipeline.predict_public_test()

# elapsed = perf_counter() - t_start
# print(f"\n✓ Hoàn tất suy luận tập test sau {elapsed / 60:.1f} phút!")
# print(f"✓ File submission tại: {submission_path}")

# # ĐỒNG BỘ TOÀN BỘ KẾT QUẢ VÀ LOGS VỀ GOOGLE DRIVE
# sync_to_drive()

# # Tải trực tiếp file submission về máy tính cá nhân (nếu muốn)
# if IN_COLAB:
#     from google.colab import files
#     drive_sub = DRIVE_OUTPUT_DIR / split_name / "submission.jsonl"
#     target_file = drive_sub if drive_sub.exists() else submission_path
#     print(f"▶ Đang tải {target_file} về máy...")
#     files.download(str(target_file))

In [9]:
import inspect
from time import perf_counter

N_TEST = 10   # số câu chạy thử; đặt None để chạy toàn bộ tập

t_start = perf_counter()
print(f"=== BẮT ĐẦU SUY LUẬN TRÊN TẬP {split_name} (N_TEST={N_TEST}) ===", flush=True)

method = getattr(pipeline, f"predict_{split_name}", None) or pipeline.predict_public_test
params = inspect.signature(method).parameters
limit_key = next((k for k in ("limit_q", "limit", "max_questions", "n_questions") if k in params), None)

q_file = TEST_DIR / "questions.jsonl"
q_bak = q_file.with_name("questions.full.jsonl.bak")
if q_bak.exists():                      # lần trước bị ngắt giữa chừng -> khôi phục file gốc trước
    q_file.write_bytes(q_bak.read_bytes())
    q_bak.unlink()

call_kwargs, truncated = {}, False
try:
    if N_TEST:
        if limit_key:
            call_kwargs[limit_key] = N_TEST
            print(f"▶ Dùng tham số {limit_key}={N_TEST}", flush=True)
        else:
            raw = q_file.read_bytes()
            q_bak.write_bytes(raw)
            lines = [l for l in raw.decode("utf-8-sig").splitlines() if l.strip()][:N_TEST]
            q_file.write_text("\n".join(lines) + "\n", encoding="utf-8")
            truncated = True
            print(f"▶ Tạm cắt questions.jsonl còn {len(lines)} câu (sẽ khôi phục sau khi chạy)", flush=True)
    submission_path = method(**call_kwargs)
finally:
    if truncated:
        q_file.write_bytes(q_bak.read_bytes())
        q_bak.unlink()
        print("↺ Đã khôi phục questions.jsonl gốc", flush=True)

elapsed = perf_counter() - t_start
print(f"\n✓ Hoàn tất suy luận sau {elapsed / 60:.1f} phút!")
print(f"✓ File submission tại: {submission_path}")

# ĐỒNG BỘ KẾT QUẢ VÀ LOGS VỀ GOOGLE DRIVE
sync_to_drive()

# Chỉ tải về máy khi chạy toàn bộ tập
if IN_COLAB and not N_TEST:
    from google.colab import files
    drive_sub = DRIVE_OUTPUT_DIR / split_name / "submission.jsonl"
    target_file = drive_sub if drive_sub.exists() else submission_path
    print(f"▶ Đang tải {target_file} về máy...")
    files.download(str(target_file))

=== BẮT ĐẦU SUY LUẬN TRÊN TẬP public_test (N_TEST=10) ===
▶ Tạm cắt questions.jsonl còn 10 câu (sẽ khôi phục sau khi chạy)
Validated 14 DuckDB databases
[test] sẽ chạy 10 câu / 8 business (tổng split: 10 câu / 8 business)
[test_qwen_local_full] cache hợp lệ đã nạp: 0 câu
[sql] q_008fdebb6ec9 (dev_002): chuẩn bị DuckDB/catalog
[direct-ingest] dev_002/README.md: Unsupported source format
[sql] q_008fdebb6ec9: rewrite và sinh SQL
[hf] prefill 627 tokens in 0.36s (1761.3 tok/s)
[hf] done 30 tokens in 0.97s (31.0 tok/s)
[hf] prefill 12013 tokens in 0.33s (36877.1 tok/s)
[hf] done 41 tokens in 1.36s (30.2 tok/s)
[hf] prefill 12032 tokens in 0.08s (143989.3 tok/s)
[hf] done 39 tokens in 1.32s (29.5 tok/s)
[hf] prefill 12013 tokens in 0.08s (145821.5 tok/s)
[hf] done 38 tokens in 1.26s (30.1 tok/s)
[hf] prefill 12032 tokens in 0.08s (145365.2 tok/s)
[hf] done 33 tokens in 1.09s (30.4 tok/s)
[hf] prefill 12032 tokens in 0.08s (145399.9 tok/s)
[hf] done 28 tokens in 0.92s (30.5 tok/s)
[hf] prefi

## 8. Kiểm Tra & Thống Kê Kết Quả Đã Lưu Trên Google Drive

In [10]:
import pandas as pd

print(f"=== DANH SÁCH FILE VÀ LOG ĐÃ LƯU TRONG GOOGLE DRIVE ===\n")
saved_files = sorted(DRIVE_OUTPUT_DIR.rglob("*"))
for f in saved_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"✓ {f.relative_to(DRIVE_OUTPUT_DIR)} ({size_kb:.1f} KB)")

# Kiểm tra nội dung file submission.jsonl
sub_file = DRIVE_OUTPUT_DIR / split_name / "submission.jsonl"
if not sub_file.exists():
    sub_file = LOCAL_RUN_DIR / split_name / "submission.jsonl"

if sub_file.exists():
    rows = [json.loads(line) for line in sub_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    df_sub = pd.DataFrame(rows)
    print(f"\n✓ Tổng số câu trong submission: {len(df_sub)}")
    print("\nPhân bố loại đáp án dự đoán:")
    print(df_sub['type'].value_counts())
    print("\n10 câu dự đoán đầu tiên:")
    display(df_sub.head(10))
else:
    print("⚠️ Chưa tìm thấy submission.jsonl!")

=== DANH SÁCH FILE VÀ LOG ĐÃ LƯU TRONG GOOGLE DRIVE ===

✓ databases/public_test/dev_001/current/data.duckdb (2316.0 KB)
✓ databases/public_test/dev_002/current/data.duckdb (5644.0 KB)
✓ databases/public_test/dev_003/current/data.duckdb (8460.0 KB)
✓ databases/public_test/dev_004/current/data.duckdb (7692.0 KB)
✓ databases/public_test/dev_005/current/data.duckdb (7692.0 KB)
✓ databases/public_test/dev_006/current/data.duckdb (9996.0 KB)
✓ databases/public_test/dev_007/current/data.duckdb (5900.0 KB)
✓ databases/public_test/dev_008/current/data.duckdb (7948.0 KB)
✓ databases/public_test/dev_009/current/data.duckdb (1036.0 KB)
✓ databases/public_test/dev_010/current/data.duckdb (2828.0 KB)
✓ databases/public_test/dev_011/current/data.duckdb (1036.0 KB)
✓ databases/public_test/dev_012/current/data.duckdb (4620.0 KB)
✓ databases/public_test/dev_013/current/data.duckdb (5644.0 KB)
✓ databases/public_test/dev_014/current/data.duckdb (6412.0 KB)
✓ public_test/logs/llm_cache_test_qwen_local_fu

,id,type,reason
0,q_008fdebb6ec9,unanswerable,The question cannot be answered because the da...
1,q_009187cd2960,unanswerable,Table 'v_sales' does not exist in the catalog.
2,q_00bf8df3ea0b,unanswerable,Dữ liệu không đầy đủ để tính doanh thu quý. Ch...
3,q_00c4e24fa228,unanswerable,Cột 'product_name' không tồn tại trong bảng v_...
4,q_013b4ff35811,unanswerable,The table v_sales does not exist in the catalog.
5,q_014cdcc1da6e,unanswerable,Dữ liệu không cung cấp thông tin về doanh thu ...
6,q_01ac0fc9f2a2,unanswerable,Công cụ phân tích không tìm thấy bảng v_sales.
7,q_01ba91ff94c6,unanswerable,"Vào thời điểm câu hỏi được đặt, bảng v_sales c..."
8,q_01d1099489a0,unanswerable,Table with name v_sales does not exist!
9,q_01d1fc600b26,unanswerable,Dữ liệu không có thông tin về doanh thu của Bi...


In [11]:
import json, collections
recs = []
for f in LOCAL_RUN_DIR.rglob("*.jsonl"):
    for line in f.read_text(encoding="utf-8", errors="ignore").splitlines():
        try:
            d = json.loads(line)
        except Exception:
            continue
        log = d.get("log", d) if isinstance(d, dict) else None
        if isinstance(log, dict) and "samples" in log and "errors" in log:
            recs.append(log)
print(len(recs), "bản ghi log")

reasons, errs, intents = collections.Counter(), collections.Counter(), collections.Counter()
for d in recs:
    intents[d.get("intent")] += 1
    for s in d.get("samples") or []:
        if s and s.get("type") == "unanswerable":
            reasons[str(s.get("reason"))[:160]] += 1
    for e in d.get("errors") or []:
        errs[str(e.get("error"))[:160]] += 1
print("--- intent ---", dict(intents))
print("--- Lý do unanswerable ---")
for r, c in reasons.most_common(10): print(c, r)
print("--- Lỗi sinh/thực thi ---")
for e, c in errs.most_common(10): print(c, e)

20 bản ghi log
--- intent --- {'sql': 20}
--- Lý do unanswerable ---
2 Vấn đề không thể trả lời bởi dữ liệu hiện có.
2 The question cannot be answered because the data for the month of November 2025 does not exist in the provided dataset.
2 Catalog Error: Table with name v_sales does not exist!
2 Table 'v_sales' does not exist in the catalog.
2 Dữ liệu không đủ để tính doanh thu theo quý cụ thể.
2 Dữ liệu không đầy đủ để tính doanh thu quý. Chỉ có dữ liệu cho tháng 10/2025.
2 Dữ liệu phát hành không đủ để trả lời câu hỏi. Không có dữ liệu bán hàng trong tháng 6/2026.
2 Cột 'product_name' không tồn tại trong bảng v_sales. Cột tương ứng là 'Tên hàng'.
2 View v_sales không tồn tại trong catalog!
2 The table v_sales does not exist in the catalog.
--- Lỗi sinh/thực thi ---
6 CatalogException: Catalog Error: Table with name v_sales does not exist!
Did you mean "pg_tables"?

LINE 1: SELECT ROUND(SUM(amount), 0) FROM "v_sales" WHERE or
4 GenerationError: sql: invalid JSON after 3 attempts: 'sq